In [1]:
from datasets import load_dataset

ds = load_dataset("OpenLLM-Ro/ro_wiki")
print(ds)
print(ds["train"][0])

c:\Users\tirid\miniconda3\envs\ro-graph-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tirid\miniconda3\envs\ro-graph-rag\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tirid\.cache\huggingface\hub\datasets--OpenLLM-Ro--ro_wiki. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order t

DatasetDict({
    train: Dataset({
        features: ['page'],
        num_rows: 122940
    })
    validation: Dataset({
        features: ['page'],
        num_rows: 780
    })
    test: Dataset({
        features: ['page'],
        num_rows: 779
    })
})
{'page': 'Sucul pancreatic este un lichid alcalin (pH 8) care conține enzime, proenzime și electroliți pentru digestia carbohidraților, proteinelor, grăsimilor și acizilor nucleici; 1,5-2 l sunt secretați zilnic de acinii exocrini ai celulelor epiteliale pancreatice, care însumează 98% din masa pancreasului. Sucul pancreatic este un lichid clar, vâscos, având un pH apropiat de 8. Bazicitatea este conferită de concentrația mare de ioni bicarbonat pe care îi conține. Sucul pancreatic este izoton în raport cu lichidul extracelular. În condiții normale, pancreasul secretă 1,5-2 l de suc pancreatic pe zi. În compoziția sucului pancreatic intră substanțe anorganice și organice. Dintre substanțele anorganice concentrațiile de Na+ și K+ sun

In [2]:
from datasets import load_dataset
import json
from pathlib import Path

out_dir = Path("data/rowiki")
out_dir.mkdir(parents=True, exist_ok=True)

ds = load_dataset("OpenLLM-Ro/ro_wiki")

for split in ["train", "validation", "test"]:
    out_path = out_dir / f"{split}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for row in ds[split]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved to", out_dir)

Saved to data\rowiki


In [3]:
for split in ["train", "validation", "test"]:
    path = Path(f"data/rowiki/{split}.jsonl")
    print(f"\n=== {split} ===")
    with open(path, "r", encoding="utf-8") as f:
        first = json.loads(next(f))
        print("Keys:", first.keys())
        for k, v in first.items():
            if isinstance(v, str):
                print(f"{k}: {v[:300]!r}")
            else:
                print(f"{k}: {v}")


=== train ===
Keys: dict_keys(['page'])
page: 'Sucul pancreatic este un lichid alcalin (pH 8) care conține enzime, proenzime și electroliți pentru digestia carbohidraților, proteinelor, grăsimilor și acizilor nucleici; 1,5-2 l sunt secretați zilnic de acinii exocrini ai celulelor epiteliale pancreatice, care însumează 98% din masa pancreasului. '

=== validation ===
Keys: dict_keys(['page'])
page: 'Franța are un sistem politic multipartit cu numeroase partide, dintre care nici unul nu a avut ocazia să guverneze singur, fiind necesară formarea de coaliții.\n Din anii 1980 guvernul a alternat între două coaliții principale:\n Următoarele partide au o reprezentare unifiromă la nivel național.\n Part'

=== test ===
Keys: dict_keys(['page'])
page: 'Fondată în 1926 de către Radio Corporation of America (RCA), NBC este cea mai veche rețea de difuzare majoră din Statele Unite.\n La acel moment societatea-mamă a RCA era General Electric (GE).\n În 1930, GE a fost forțată să vândă companiile ca

In [4]:
INPUT_DIR = Path("data/rowiki")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

splits = ["train", "validation", "test"]
all_rows = []

for split in splits:
    path = INPUT_DIR / f"{split}.jsonl"
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            row["_split"] = split
            all_rows.append(row)

print(f"Loaded {len(all_rows)} total rows")

with open(OUTPUT_DIR / "rowiki_all.jsonl", "w", encoding="utf-8") as f:
    for row in all_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved merged corpus.")

Loaded 124499 total rows
Saved merged corpus.


In [5]:
import re

INPUT_PATH = Path("data/processed/rowiki_all.jsonl")
OUTPUT_PATH = Path("data/processed/rowiki_clean.jsonl")

MIN_CHARS = 500
MAX_CHARS = 12000

def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

seen = set()
kept = 0
dropped_short = 0
dropped_long = 0
dropped_dupe = 0

with open(INPUT_PATH, "r", encoding="utf-8") as fin, open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for idx, line in enumerate(fin):
        row = json.loads(line)

        text = row.get("page") or row.get("text") or row.get("content") or ""
        text = clean_text(text)

        if len(text) < MIN_CHARS:
            dropped_short += 1
            continue
        if len(text) > MAX_CHARS:
            dropped_long += 1
            continue

        if text in seen:
            dropped_dupe += 1
            continue
        seen.add(text)

        out = {
            "doc_id": f"rowiki_{kept:05d}",
            "source": "rowiki",
            "split": row.get("_split"),
            "text": text,
            "char_len": len(text),
        }

        fout.write(json.dumps(out, ensure_ascii=False) + "\n")
        kept += 1

print("Kept:", kept)
print("Dropped short:", dropped_short)
print("Dropped long:", dropped_long)
print("Dropped duplicates:", dropped_dupe)

Kept: 85977
Dropped short: 35017
Dropped long: 3461
Dropped duplicates: 44


In [ ]:
# Select candidates based on keyword presence for demo purposes

INPUT_PATH = Path("data/processed/rowiki_clean.jsonl")
OUTPUT_PATH = Path("data/processed/corpus_candidates_admin.jsonl")

keywords = [
    "românia", "guvern", "minister", "primărie", "parlament",
    "județ", "municipiu", "oraș", "universitate", "instituție"
]

matches = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        text_lower = row["text"].lower()

        score = sum(1 for kw in keywords if kw in text_lower)
        if score >= 2:
            row["keyword_score"] = score
            matches.append(row)

matches = sorted(matches, key=lambda x: x["keyword_score"], reverse=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for row in matches[:300]:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {min(len(matches), 300)} candidates")

Saved 300 candidates


In [7]:
import spacy

INPUT_PATH = Path("data/processed/corpus_candidates_admin.jsonl")
CORPUS_OUT = Path("data/processed/corpus_100.jsonl")
CHUNKS_OUT = Path("data/processed/chunks_100.jsonl")

MAX_DOCS = 100
MAX_CHARS_PER_CHUNK = 1200
MIN_CHARS_PER_CHUNK = 400

nlp = spacy.load("ro_core_news_lg")

def load_first_n_jsonl(path, n):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rows.append(json.loads(line))
    return rows

def normalize_text(text: str) -> str:
    return " ".join(text.split()).strip()

def chunk_with_spacy(text, max_chars=1200, min_chars=400):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    chunks = []
    current = []

    for sent in sentences:
        candidate = " ".join(current + [sent]).strip()

        if len(candidate) <= max_chars:
            current.append(sent)
        else:
            if current:
                chunk = " ".join(current).strip()
                if len(chunk) >= min_chars:
                    chunks.append(chunk)
                elif chunks:
                    chunks[-1] = (chunks[-1] + " " + chunk).strip()
                else:
                    chunks.append(chunk)

            if len(sent) > max_chars:
                start = 0
                while start < len(sent):
                    end = start + max_chars
                    piece = sent[start:end].strip()
                    if piece:
                        chunks.append(piece)
                    if end >= len(sent):
                        break
                    start = end - 150
                current = []
            else:
                current = [sent]

    if current:
        chunk = " ".join(current).strip()
        if len(chunk) >= min_chars:
            chunks.append(chunk)
        elif chunks:
            chunks[-1] = (chunks[-1] + " " + chunk).strip()
        else:
            chunks.append(chunk)

    return chunks

def main():
    docs = load_first_n_jsonl(INPUT_PATH, MAX_DOCS)
    CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

    cleaned_docs = []
    for i, row in enumerate(docs):
        text = normalize_text(row.get("text", ""))
        cleaned_docs.append({
            "doc_id": f"rowiki_admin_{i:03d}",
            "source": "rowiki",
            "topic": "admin",
            "text": text,
            "char_len": len(text)
        })

    with open(CORPUS_OUT, "w", encoding="utf-8") as f:
        for row in cleaned_docs:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    total_chunks = 0
    with open(CHUNKS_OUT, "w", encoding="utf-8") as f:
        for row in cleaned_docs:
            chunks = chunk_with_spacy(
                row["text"],
                max_chars=MAX_CHARS_PER_CHUNK,
                min_chars=MIN_CHARS_PER_CHUNK
            )

            for j, chunk in enumerate(chunks):
                out = {
                    "chunk_id": f"{row['doc_id']}_chunk_{j:03d}",
                    "doc_id": row["doc_id"],
                    "source": row["source"],
                    "topic": row["topic"],
                    "text": chunk,
                    "char_len": len(chunk)
                }
                f.write(json.dumps(out, ensure_ascii=False) + "\n")
                total_chunks += 1

    print(f"Saved {len(cleaned_docs)} docs to {CORPUS_OUT}")
    print(f"Saved {total_chunks} chunks to {CHUNKS_OUT}")

if __name__ == "__main__":
    main()

Saved 100 docs to data\processed\corpus_100.jsonl
Saved 658 chunks to data\processed\chunks_100.jsonl
